Calculate the inner product of two vectors with floating point numbers and time the calculation.

In [ ]:
!pip install pyopencl

In [ ]:
import numpy as np
import pyopencl as cl
import pyopencl.array as cl_array
import time

In [ ]:
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
print("Platform name:", platform.name)
print("Device name:", device.name)
print("Maximum work group size:", device.max_work_group_size)

In [ ]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)

In the following, we wish to divide the data in smaller workgroups. For this purpose, we will define the workgroup size and the number of data points. Later, the program can only be used when the workgroup size is a divisor of the number of data points. If this is not the case, one needs to pad the data array with dummy variables.

In [ ]:
workgroup_size = 2**6
n_workgroups = 8
vector_size = workgroup_size * n_workgroups
print("Workgroup size:", workgroup_size)
print("Vector size:", vector_size)

Calculating the sum of a vector is a recursive process and, therefore, not well suited for a SIMD architecture. Different approaches can be used to sum a vector. In most cases, only the part of the vector that is with the same workgroup will taken into account for a sum. Then, the sum of the partial sums will be calculated on the CPU. This is more efficient since in most cases the number of different workgroups is limited. Alternatively, one can use global memory to sum the partial sums on the GPU.

The first option is to let one thread sum all values. This is slow because no parallelism is used.

The second option is to implement a binary tree, which has parallelism. The implementation below only works for a work group size that is a power of two. Since if-else statements are slow on the GPU, in case of a workgroup that is not a power of two, it is better to pad the input vector with zero elements until reaching a power of two. Also, the barrier is necessary to avoid race conditions between threads.

In [ ]:
kernel = """
__kernel void dot1(__global double *vec1,
                   __global double *vec2,
                   __global double *partial_sums)
{
  int group_size = get_local_size(0);
  int local_id = get_local_id(0);
  int group_id = get_group_id(0);
  int global_id = get_global_id(0);

  vec1[global_id] *= vec2[global_id];

  if (local_id == 0){
    double sum = 0.0;
    for(int i = 0; i < group_size; i++){
      sum += vec1[global_id + i];
    }
    partial_sums[group_id] = sum;
  }
}
__kernel void dot2(__global double *vec1,
                   __global double *vec2,
                   __global double *partial_sums)
{
  int group_size = get_local_size(0);
  int local_id = get_local_id(0);
  int group_id = get_group_id(0);
  int global_id = get_global_id(0);

  vec1[global_id] *= vec2[global_id];

  int step = 2;
  while (step <= group_size) {
    if (local_id % step == 0) {
      vec1[global_id] += vec1[global_id + step / 2];
    }
    barrier(CLK_GLOBAL_MEM_FENCE);
    step *= 2;
  }
  if (local_id == 0){
    partial_sums[group_id] = vec1[global_id];
  }
}
"""

In [ ]:
prg = cl.Program(ctx, kernel).build()

The kernel calculates the partial sums within the workgroup. The final sum will be calculated on the CPU, so we need to create an output array with a size of the number of workgroups.

The input variables of the program are as follows:
 1. the command queue
 1. the global size of the data array, possibly multi-dimensional
 1. the size of the workgroups, which needs to divide the global size evenly
 1. the buffers for the input variables of the kernel function

In [ ]:
np.random.seed(0)
np_vector1 = np.random.rand(vector_size)
np_vector2 = np.random.rand(vector_size)

In [ ]:
# Calculate the dot product with Numpy.
time_start = time.time()
vector_sum_np = np.inner(np_vector1, np_vector2)
np_time = time.time() - time_start

In [ ]:
# Calculate the sum of the vector with OpenCL, on one thread per workgroup.

time_start = time.time()

cl_vector1 = cl_array.to_device(queue, np_vector1)
cl_vector2 = cl_array.to_device(queue, np_vector2)
cl_partial_sums1 = cl_array.empty(queue, n_workgroups, dtype=np.float64)

event = prg.dot1(queue, (vector_size,), (workgroup_size,),
                 cl_vector1.data, cl_vector2.data, cl_partial_sums1.data)

np_partial_sums1 = cl_partial_sums1.get()
vector_sum_cl1 = np.sum(np_partial_sums1)

cl_time1 = time.time() - time_start
cl_time1_kernel = 1e-9 * (event.profile.end - event.profile.start)

In [ ]:
# Calculate the sum of the vector with OpenCL, with a binary tree per workgroup.

time_start = time.time()

cl_vector1 = cl_array.to_device(queue, np_vector1)
cl_vector2 = cl_array.to_device(queue, np_vector2)
cl_partial_sums2 = cl_array.empty(queue, n_workgroups, dtype=np.float64)

event = prg.dot2(queue, (vector_size,), (workgroup_size,),
                 cl_vector1.data, cl_vector2.data, cl_partial_sums2.data)

np_partial_sums2 = cl_partial_sums2.get()
vector_sum_cl2 = np.sum(np_partial_sums2)

cl_time2 = time.time() - time_start
cl_time2_kernel = 1e-9 * (event.profile.end - event.profile.start)

In [ ]:
print("Result of Numpy   :", vector_sum_np)
print("Result of OpenCL 1:", vector_sum_cl1)
print("Result of OpenCL 2:", vector_sum_cl2)

In [ ]:
print("Elapsed time Numpy   :", np_time)
print("Elapsed time OpenCL 1:", cl_time1)
print("   of which in kernel:", cl_time1_kernel)
print("Elapsed time OpenCL 2:", cl_time2)
print("   of which in kernel:", cl_time2_kernel)